# Entity Resolution and Golden Record Laboratory

## Notebook 01: Data Acquisition and Initial Audit

**Author:** Kazeem Wale Balogun  
**Project:** Building a Trusted Customer View  
**Purpose:** Load the FEBRL dataset, inspect its structure and conduct an initial data-quality assessment.


In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)


Python executable:
c:\Users\DELL\venvs\entity-resolution-golden-record-lab\Scripts\python.exe

Python version:
3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]


In [2]:
import pandas as pd
import numpy as np
import matplotlib
import sklearn
import recordlinkage
import ipykernel

print("All project libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Recordlinkage version: {recordlinkage.__version__}")
print(f"IPykernel version: {ipykernel.__version__}")

All project libraries imported successfully.
Pandas version: 2.3.3
NumPy version: 2.5.1
Scikit-learn version: 1.9.0
Recordlinkage version: 0.16
IPykernel version: 7.3.0


## 1. Data Acquisition

This project begins with FEBRL Dataset 1, a fictitious person-record dataset designed for testing record-linkage and duplicate-detection methods.

The dataset contains original records and deliberately corrupted duplicate records. The known matching pairs will be retained as the ground truth for evaluating the performance of rule-based and machine-learning approaches.

In [3]:
from recordlinkage.datasets import load_febrl1

print("FEBRL dataset loader imported successfully.")

FEBRL dataset loader imported successfully.


In [4]:
# Load FEBRL Dataset 1 and its known duplicate relationships
febrl_records, true_links = load_febrl1(return_links=True)

print("FEBRL Dataset 1 loaded successfully.")

FEBRL Dataset 1 loaded successfully.


In [5]:
print("Records object type:")
print(type(febrl_records))

print("\nTrue-links object type:")
print(type(true_links))

Records object type:
<class 'pandas.core.frame.DataFrame'>

True-links object type:
<class 'pandas.core.indexes.multi.MultiIndex'>


### 1.1 Dataset Dimensions

In [6]:
number_of_records, number_of_columns = febrl_records.shape
number_of_true_matches = len(true_links)

print(f"Number of records: {number_of_records:,}")
print(f"Number of columns: {number_of_columns}")
print(f"Number of known duplicate pairs: {number_of_true_matches:,}")

Number of records: 1,000
Number of columns: 10
Number of known duplicate pairs: 500


In [7]:
print("Dataset columns:")

for column_number, column_name in enumerate(
    febrl_records.columns,
    start=1
):
    print(f"{column_number}. {column_name}")

Dataset columns:
1. given_name
2. surname
3. street_number
4. address_1
5. address_2
6. suburb
7. postcode
8. state
9. date_of_birth
10. soc_sec_id


### 1.2 Initial Record Inspection

In [8]:
febrl_records.head()

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-223-org,NaN,waller,6,tullaroop street,willaroo,st james,4011,wa,19081209,6988048
rec-122-org,lachlan,berry,69,giblin street,killarney,bittern,4814,qld,19990219,7364009
rec-373-org,deakin,sondergeld,48,goldfinch circuit,kooltuo,canterbury,2776,vic,19600210,2635962
rec-10-dup-0,kayla,harrington,NaN,maltby circuit,coaling,coolaroo,3465,nsw,19150612,9004242
rec-227-org,luke,purdon,23,ramsay place,mirani,garbutt,2260,vic,19831024,8099933


In [9]:
print("Index name:")
print(febrl_records.index.name)

print("\nFirst ten record identifiers:")
print(febrl_records.index[:10].tolist())

Index name:
rec_id

First ten record identifiers:
['rec-223-org', 'rec-122-org', 'rec-373-org', 'rec-10-dup-0', 'rec-227-org', 'rec-6-dup-0', 'rec-190-dup-0', 'rec-294-org', 'rec-206-dup-0', 'rec-344-org']


### 1.3 Known Duplicate Relationships

In [10]:
print("First ten known duplicate pairs:")

for record_id_1, record_id_2 in list(true_links[:10]):
    print(f"{record_id_1}  <->  {record_id_2}")

First ten known duplicate pairs:
rec-344-dup-0  <->  rec-344-org
rec-251-org  <->  rec-251-dup-0
rec-335-dup-0  <->  rec-335-org
rec-23-dup-0  <->  rec-23-org
rec-382-org  <->  rec-382-dup-0
rec-70-dup-0  <->  rec-70-org
rec-291-org  <->  rec-291-dup-0
rec-179-org  <->  rec-179-dup-0
rec-452-dup-0  <->  rec-452-org
rec-383-org  <->  rec-383-dup-0


In [11]:
# Select the first known duplicate pair
first_match_id, second_match_id = true_links[0]

print("First known duplicate pair:")
print(first_match_id, "<->", second_match_id)

febrl_records.loc[[first_match_id, second_match_id]]

First known duplicate pair:
rec-344-dup-0 <-> rec-344-org


,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-344-dup-0,NaN,stephenson,52,NaN,north stirilng downs,coolaroo,2259,qld,19630521,1797144
rec-344-org,NaN,julius,52,florey drive,north stirling downs,coolaroo,2259,qld,19630521,1797144


### Initial Observation

The selected record pair represents the same underlying person but contains differences across one or more identifying attributes. These variations illustrate why exact matching alone may fail to identify duplicate customer records.

Further analysis will determine which fields are complete, which types of variations occur most frequently and which attributes provide the strongest evidence for entity resolution.